# [STARTER] Udaplay Project

## Part 01 - Offline RAG

In this part of the project, you'll build your VectorDB using Chroma.

The data is inside folder `project/starter/games`. Each file will become a document in the collection you'll create.
Example.:
```json
{
  "Name": "Gran Turismo",
  "Platform": "PlayStation 1",
  "Genre": "Racing",
  "Publisher": "Sony Computer Entertainment",
  "Description": "A realistic racing simulator featuring a wide array of cars and tracks, setting a new standard for the genre.",
  "YearOfRelease": 1997
}
```


### Setup

In [48]:
# Only needed for Udacity workspace

import importlib.util
import sys

# Check if 'pysqlite3' is available before importing
if importlib.util.find_spec("pysqlite3") is not None:
    import pysqlite3
    sys.modules['sqlite3'] = sys.modules.pop('pysqlite3')

In [49]:
import os
import json
import chromadb
from chromadb.utils import embedding_functions
from dotenv import load_dotenv

In [50]:
# TODO: Create a .env file with the following variables
# OPENAI_API_KEY="YOUR_KEY"
# CHROMA_OPENAI_API_KEY="YOUR_KEY"
# TAVILY_API_KEY="YOUR_KEY"

In [51]:
# TODO: Load environment variables
load_dotenv("config.env")
import os
print(os.getenv("OPENAI_API_KEY"))
print(os.getenv("CHROMA_OPENAI_API_KEY"))

voc-33023444816886550949156a0d3f7424d9f8.97050777
voc-33023444816886550949156a0d3f7424d9f8.97050777


### VectorDB Instance

In [52]:
# TODO: Instantiate your ChromaDB Client
# Choose any path you want
chroma_client = chromadb.PersistentClient(path="chromadb")

### Collection

In [53]:
# TODO: Pick one embedding function
# If picking something different than openai, 
# make sure you use the same when loading it
embedding_fn = embedding_functions.OpenAIEmbeddingFunction(
    api_key=os.getenv("OPENAI_API_KEY"),     
    model_name="text-embedding-ada-002"
 
)

In [54]:
# TODO: Create a collection
# Choose any name you want
collection = chroma_client.get_or_create_collection(
    name="udaplay",
    embedding_function=embedding_fn
)


### Add documents

In [55]:
documents = []
metadatas = []
ids = []

games_dir = "games"
for filename in sorted(os.listdir(games_dir)):
    if filename.endswith(".json"):
        filepath = os.path.join(games_dir, filename)
        with open(filepath, "r") as f:
            game = json.load(f)

        documents.append(game.get("Description", ""))
        metadatas.append({
            "name": game.get("Name", ""),
            "platform": game.get("Platform", ""),
            "genre": game.get("Genre", ""),
            "publisher": game.get("Publisher", ""),
            "year": str(game.get("YearOfRelease", ""))
        })
        ids.append(filename.replace(".json", ""))

collection.add(
    documents=documents,
    metadatas=metadatas,
    ids=ids
)

print(f"Added {len(documents)} games to the collection.")

Added 15 games to the collection.


In [56]:
results = collection.query(
    query_texts=["a realistic racing game"],
    n_results=3
)

for i, doc in enumerate(results["documents"][0]):
    name = results["metadatas"][0][i]["name"]
    print(f"{i+1}. {name}: {doc[:80]}...")

1. Gran Turismo: A realistic racing simulator featuring a wide array of cars and tracks, setting ...
2. Gran Turismo 5: A comprehensive racing simulator featuring a vast selection of vehicles and trac...
3. Mario Kart 8 Deluxe: An enhanced version of Mario Kart 8, featuring new characters, tracks, and impro...
